# SABR volatility smiles and calibration

The SABR (Stochastic Alpha-Beta-Rho) model produces an implied-volatility smile from four parameters and the Hagan (2002) expansion. `finstack_quant.valuations` exposes:

- `SabrParameters(alpha, beta, nu, rho, shift=None)` -- the model parameters,
- `SabrModel` -- single-point implied vol for any `(forward, strike, t)`,
- `SabrSmile` -- a smile at a fixed `(forward, t)` across strikes, with arbitrage diagnostics,
- `SabrCalibrator` -- fits `alpha`, `nu`, `rho` to market quotes (`beta` is held fixed).

### Parameter roles

- **alpha** -- overall (ATM) volatility level,
- **beta** -- backbone / vol regime: `1.0` lognormal (equity Black vols), `0.5` common for rates, `0.0` normal (Bachelier) vols,
- **nu** -- vol-of-vol, controlling smile curvature,
- **rho** -- spot/vol correlation, controlling skew.

## Imports

In [1]:
import pandas as pd

from finstack_quant.valuations import (
    SabrParameters,
    SabrModel,
    SabrSmile,
    SabrCalibrator,
)

FORWARD = 100.0
EXPIRY = 1.0  # years

## Parameters and a single implied vol

`SabrModel.implied_vol(forward, strike, t)` returns the Black implied volatility (a decimal, e.g. `0.20` = 20%) for `beta > 0`. There are also convenience defaults for equity and rates.

In [2]:
params = SabrParameters(0.20, 1.0, 0.30, -0.20)  # alpha, beta, nu, rho
model = SabrModel(params)

print(f"ATM implied vol      : {model.implied_vol(FORWARD, FORWARD, EXPIRY):.4%}")
print(f"OTM call vol (K=120) : {model.implied_vol(FORWARD, 120.0, EXPIRY):.4%}")
print(f"OTM put  vol (K=80)  : {model.implied_vol(FORWARD, 80.0, EXPIRY):.4%}")

eq, rt = SabrParameters.equity_default(), SabrParameters.rates_default()
print(f"\nequity_default: alpha={eq.alpha}, beta={eq.beta}, nu={eq.nu}, rho={eq.rho}")
print(f"rates_default : alpha={rt.alpha}, beta={rt.beta}, nu={rt.nu}, rho={rt.rho}")

ATM implied vol      : 20.0810%
OTM call vol (K=120) : 19.7789%
OTM put  vol (K=80)  : 21.0694%

equity_default: alpha=0.2, beta=1.0, nu=0.3, rho=-0.2
rates_default : alpha=0.02, beta=0.5, nu=0.3, rho=0.0


## Building a smile

`SabrSmile` fixes `(forward, t)` and evaluates across strikes. `generate_smile(strikes)` returns the vols in order; `atm_vol()` returns the at-the-forward level.

In [3]:
strikes = [80.0, 85.0, 90.0, 95.0, 100.0, 105.0, 110.0, 115.0, 120.0]
smile = SabrSmile(params, FORWARD, EXPIRY)
vols = smile.generate_smile(strikes)

print(f"ATM vol: {smile.atm_vol():.4%}")
smile_df = pd.DataFrame({"strike": strikes, "implied_vol": vols})
smile_df["moneyness"] = smile_df["strike"] / FORWARD
smile_df

ATM vol: 20.0810%


,strike,implied_vol,moneyness
0,80.0,0.210694,0.80
1,85.0,0.207440,0.85
2,90.0,0.204734,0.90
3,95.0,0.202537,0.95
4,100.0,0.200810,1.00
5,105.0,0.199512,1.05
6,110.0,0.198602,1.10
7,115.0,0.198041,1.15
8,120.0,0.197789,1.20


## Effect of the parameters

`rho` tilts the smile into a skew (more negative -> steeper down-strike skew); `nu` (vol-of-vol) deepens the smile curvature. Holding `alpha` and `beta` fixed (values shown as decimal vols):

In [4]:
def smile_for(alpha, beta, nu, rho):
    return SabrSmile(SabrParameters(alpha, beta, nu, rho), FORWARD, EXPIRY).generate_smile(strikes)

effects = pd.DataFrame({"strike": strikes})
effects["base (rho=-0.2, nu=0.3)"] = smile_for(0.20, 1.0, 0.30, -0.20)
effects["steeper skew (rho=-0.5)"] = smile_for(0.20, 1.0, 0.30, -0.50)
effects["more curvature (nu=0.6)"] = smile_for(0.20, 1.0, 0.60, -0.20)
effects.set_index("strike").round(4)

,"base (rho=-0.2, nu=0.3)",steeper skew (rho=-0.5),more curvature (nu=0.6)
strike,,,
80.0,0.2107,0.2180,0.2294
85.0,0.2074,0.2126,0.2208
90.0,0.2047,0.2078,0.2138
95.0,0.2025,0.2034,0.2083
100.0,0.2008,0.1994,0.2044
105.0,0.1995,0.1959,0.2022
110.0,0.1986,0.1928,0.2013
115.0,0.1980,0.1900,0.2018
120.0,0.1978,0.1877,0.2034


## Calibrating to market quotes

`SabrCalibrator.calibrate(forward, strikes, market_vols, t, beta)` fits `alpha`, `nu`, and `rho` while holding `beta` fixed. Here we generate a synthetic market from known parameters and recover them.

In [5]:
market_strikes = [80.0, 90.0, 100.0, 110.0, 120.0]
market_vols = smile.generate_smile(market_strikes)  # synthetic quotes from the known params

fitted = SabrCalibrator().calibrate(FORWARD, market_strikes, market_vols, EXPIRY, 1.0)  # beta fixed at 1.0

print(f"true : alpha={params.alpha:.4f}  nu={params.nu:.4f}  rho={params.rho:.4f}")
print(f"fit  : alpha={fitted.alpha:.4f}  nu={fitted.nu:.4f}  rho={fitted.rho:.4f}  (beta fixed at {fitted.beta})")

refit = SabrSmile(fitted, FORWARD, EXPIRY).generate_smile(market_strikes)
pd.DataFrame({"strike": market_strikes, "market_vol": market_vols, "fitted_vol": refit})

true : alpha=0.2000  nu=0.3000  rho=-0.2000
fit  : alpha=0.1991  nu=0.3392  rho=-0.1603  (beta fixed at 1.0)


,strike,market_vol,fitted_vol
0,80.0,0.210694,0.210702
1,90.0,0.204734,0.204307
2,100.0,0.200810,0.200434
3,110.0,0.198602,0.198694
4,120.0,0.197789,0.198655


The fit recovers the smile within a fraction of a vol point. Note `rho` is only weakly identified on symmetric strike sets, so validate on smile shape rather than the raw `rho` value.

## Rates: beta = 0.5 and shifted SABR for negative rates

With `beta = 0.5` the inputs are decimal rates and the smile is in rate-vol units. For negative-rate environments, `calibrate_auto_shift` finds a positive displacement so the standard expansion stays well-defined.

In [6]:
rates_params = SabrParameters(0.05, 0.5, 0.40, -0.10)
rate_strikes = [0.01, 0.02, 0.03, 0.04, 0.05]
rate_smile = SabrSmile(rates_params, 0.03, 5.0).generate_smile(rate_strikes)
print("rates smile (beta=0.5):", [f"{v:.4%}" for v in rate_smile])

# Negative-rate environment: build a synthetic shifted market, then auto-shift calibrate
neg_forward = -0.005
neg_strikes = [-0.015, -0.010, -0.005, 0.0, 0.005]
neg_market = SabrSmile(SabrParameters(0.05, 0.5, 0.40, -0.10, shift=0.03), neg_forward, 1.0).generate_smile(neg_strikes)
shifted_fit = SabrCalibrator().calibrate_auto_shift(neg_forward, neg_strikes, neg_market, 1.0, 0.5)
print(f"\nauto-shift fitted shift = {shifted_fit.shift:.4f}, is_shifted = {shifted_fit.is_shifted()}")

rates smile (beta=0.5): ['48.3174%', '36.0308%', '30.6801%', '28.7745%', '28.5168%']

auto-shift fitted shift = 0.0200, is_shifted = True


## Arbitrage diagnostics

`arbitrage_diagnostics(strikes)` checks the smile for static-arbitrage violations (butterfly / call-spread monotonicity).

In [7]:
diag = smile.arbitrage_diagnostics(strikes)
print("arbitrage_free          :", diag["arbitrage_free"])
print("butterfly_violations    :", diag["butterfly_violations"])
print("monotonicity_violations :", diag["monotonicity_violations"])

arbitrage_free          : True
butterfly_violations    : []
monotonicity_violations : []


## Takeaways

- `SabrParameters(alpha, beta, nu, rho, shift=None)` plus `SabrModel.implied_vol(forward, strike, t)` give a single Black implied vol; `equity_default()` / `rates_default()` provide sensible starting points.
- `SabrSmile` evaluates a full smile at a fixed `(forward, t)`; `rho` controls skew and `nu` controls curvature.
- `SabrCalibrator.calibrate(...)` fits `alpha`, `nu`, `rho` with `beta` held fixed; `calibrate_auto_shift(...)` adds a displacement for negative-rate strikes.
- `SabrSmile.arbitrage_diagnostics(strikes)` validates the fitted smile for static arbitrage.